# Module 6 Data Cleaning and Verification

This notebook **imports and calls `cleaning.py`**. All cleaning rules, calculations, record checks, exports, and pipeline assertions live in that script. This notebook explains the project and displays the evidence produced by the script; it does not maintain a second cleaning implementation.

Place this notebook beside `cleaning.py` in the project root. Keep `prices.parquet`, `news.parquet`, and `filings.parquet` in `data/`. Install the existing project dependencies in your kernel environment, then **Restart Kernel and Run All**. Repeat in a fresh kernel to verify reproducibility and save the outputs. No downloads or package installations are performed here.

## 1. Project vision and question

**Project vision:** A junior equity research analyst on a technology-sector coverage team must decide which of the seven mega-cap technology stocks (AAPL, MSFT, GOOGL, AMZN, NVDA, META, and TSLA) to prioritize for this week's deep-dive research, based on relative price performance, volatility, trading-volume activity, and recent news or filing events.

**Business question:** How does company news relate to subsequent stock-price changes and trading volume among the Magnificent Seven?

**Planned metrics and comparison:** Compare subsequent observed trading-day adjusted-close returns and volume relative to a recent company baseline across news-event categories. Recent cumulative performance and volatility are supporting metrics. News classification, publication-time alignment, subsequent-return windows, and rolling baselines remain future analysis steps.

**Completed scope:** The script prepares and validates local data, historical returns, volume, and same-date joins. Its filing-day comparison is an auxiliary diagnostic, not an answer to the news question. The historical snapshot is not current-week data. Every observed price day has news, so news presence alone cannot define comparison groups.

## 2. Define one row

| Table | Row meaning | Candidate logical key |
|---|---|---|
| prices | One company and observed trading date | symbol + date |
| news | One company-date digest record, not one original article | symbol + date |
| filings | One company-date-form excerpt | symbol + date + document_type |

IDs are table-local; `_source_row` is the original zero-based row position. Repeated keys must be investigated. The script quarantines conflicting price keys and retains nonidentical event records for aggregation.

## Configure paths and import the local script

Open this notebook from the project root, or set `PROJECT_DIR` below to that directory. The explicit file import ensures that this project's `cleaning.py` is used rather than an unrelated installed module. Rerunning the import cell loads your saved script edits.

In [1]:
from pathlib import Path
import importlib.util
import json
import hashlib
import pandas as pd
from IPython.display import display
from IPython.utils.capture import capture_output

PROJECT_DIR = Path.cwd().resolve()
SCRIPT_PATH = PROJECT_DIR / 'cleaning.py'
DATA_DIR = PROJECT_DIR / 'data'
OUTPUT_DIR = PROJECT_DIR / 'cleaning_outputs'
for path in [SCRIPT_PATH, *(DATA_DIR / f'{name}.parquet' for name in ['prices', 'news', 'filings'])]:
    if not path.is_file():
        raise FileNotFoundError(f'Missing: {path}. Set PROJECT_DIR to the folder containing cleaning.py and data/.')

spec = importlib.util.spec_from_file_location('project_cleaning', SCRIPT_PATH)
cleaning = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cleaning)
if not callable(getattr(cleaning, 'run_pipeline', None)):
    raise AttributeError('cleaning.py must provide run_pipeline(data_dir=None, output_dir=None).')
print('Imported:', cleaning.__file__)
print('Input:', DATA_DIR)
print('Output:', OUTPUT_DIR)

Imported: C:\Users\yexun\OneDrive\Desktop\Python-Group-Project\cleaning.py
Input: C:\Users\yexun\OneDrive\Desktop\Python-Group-Project\data
Output: C:\Users\yexun\OneDrive\Desktop\Python-Group-Project\cleaning_outputs


## Declare rules and expectations before execution

The script declares its decision log and five expected record outcomes before applying cleaning. The next cell shows those declarations directly from the imported script, without duplicating them in this notebook. Review the rules before running the pipeline. Edit `cleaning.py` to change them.

In [2]:
import inspect
script_source = inspect.getsource(cleaning.run_pipeline)
for start_marker, end_marker in [('rules =', 'PRICE_COLS ='), ('plans =', 'def clean_all')]:
    start = script_source.find(start_marker)
    end = script_source.find(end_marker, start + 1) if start >= 0 else -1
    if start >= 0 and end > start:
        print(script_source[start:end].strip(), '\n')
    else:
        print(f'Review the declarations directly in {SCRIPT_PATH}; marker not found: {start_marker}')

rules = pd.DataFrame([
     ('scope','Keep MAG7 only','Project scope; ADBE is not bad data','row reconciliation + excluded records'),
     ('types','Parse dates/numbers; retain IDs as strings','Analysis needs comparable types','conversion failures + dtypes'),
     ('text','Trim boundaries; normalize line endings; uppercase codes','Remove formatting variation; preserve prose','text_changes.csv'),
     ('exact_duplicates','Drop repeated identical original rows, retain first','Avoid double counting identical source records','duplicates.csv + excluded_*.csv'),
     ('keys','Quarantine missing symbol/date and conflicting price keys','Cannot safely identify/join these rows','quarantine + logical duplicate counts'),
     ('missing','Keep unknown values missing; no imputation','Avoid fabricating evidence','missing_before_after.csv'),
     ('numeric_domain','Nonpositive price, negative/fractional volume, nonfinite -> missing','Invalid domain for these metrics','suspicious.csv + numeric_changes.

## Execute the shared pipeline

This is the **only cleaning call** in this notebook. The script reads local originals, applies its rules, runs checks, and writes evidence. Its CLI entry point does not execute during import because it is guarded by `if __name__ == '__main__'`.

Verbose tables are captured and displayed by topic below. On failure, captured output is shown and the original error is raised. Do not treat old output files as results of a failed run. After a failure, fix the cause and restart/run all; the evidence helper also refuses to load results unless this run succeeded.

In [3]:
pipeline_completed = False
try:
    with capture_output() as pipeline_output:
        cleaning.run_pipeline(data_dir=DATA_DIR, output_dir=OUTPUT_DIR)
except Exception:
    pipeline_output.show()
    raise
pipeline_completed = True
print(pipeline_output.stdout)
if pipeline_output.stderr:
    print(pipeline_output.stderr)
# To inspect every captured table, run: pipeline_output.show()

Output folder: C:\Users\yexun\OneDrive\Desktop\Python-Group-Project\cleaning_outputs
prices (2656, 9)
news (3888, 4)
filings (73, 6)
type_checks records: 19
text_changes records: 146
duplicates records: 3
missing_before_after records: 19
row_reconciliation records: 3
numeric_changes records: 0
Price days with news: 2324 of 2324
No no-news comparison group exists. News availability cannot explain between-group return differences here.
PASS: every return independently checked; missing-value toy example passed.
PASS: outputs match previous complete run.
Sample rows: 50
Saved cleaned data and verification evidence to C:\Users\yexun\OneDrive\Desktop\Python-Group-Project\cleaning_outputs
Complete your individual PDF and project README with your name and GitHub branch details.



In [4]:
def evidence(filename):
    if not pipeline_completed:
        raise RuntimeError('The pipeline has not completed successfully in this run.')
    path = OUTPUT_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f'Expected script output is missing: {path}')
    return pd.read_csv(path, encoding='utf-8-sig', keep_default_na=False)

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 60)

## 3. Preserve the source

The script reads local Parquet files without modifying their bytes. SHA256 identifies the exact local snapshot; inspection timestamps are not download dates. No upstream revision is inferred from filenames.

In [5]:
manifest = json.loads((OUTPUT_DIR / 'local_source_manifest.json').read_text('utf-8'))
display(pd.DataFrame(manifest).T)

,dataset,filename,sha256,bytes,rows,version,first_inspected_utc
prices,TheFinAI/Herculean,prices.parquet,736c124db22e976dbf4ec1dd65e5a983497e5890e7999f7925b343dee5c965d5,67349,2656,Local snapshot; identify exact files by SHA256. Upstream revision not assumed.,2026-10-04T20:47:28.439297+00:00
news,TheFinAI/Herculean,news.parquet,1fd109796eada02af37a15178cc150ec252e7c02bc440c3f76641eb8fab6da25,5216446,3888,Local snapshot; identify exact files by SHA256. Upstream revision not assumed.,2026-10-04T20:47:28.532238+00:00
filings,TheFinAI/Herculean,filings.parquet,822a854865651e042c9d64288785efa0f2c0fecdec2a16016ff3c2e7a5601299,600603,73,Local snapshot; identify exact files by SHA256. Upstream revision not assumed.,2026-10-04T20:47:28.544044+00:00


## 4. Inspect columns and types

The script reports original and converted dtypes and failed conversion counts. Invalid numeric parses remain missing; invalid join keys are quarantined.

In [6]:
print('type_checks.csv')
display(evidence('type_checks.csv'))

type_checks.csv


,table,column,before,after,conversion_failures
0,prices,id,int32,string,0
1,prices,symbol,object,string,0
2,prices,date,object,datetime64[ns],0
3,prices,open,float64,float64,0
4,prices,high,float64,float64,0
5,prices,low,float64,float64,0
6,prices,close,float64,float64,0
7,prices,adj_close,float64,float64,0
8,prices,volume,int64,float64,0
9,news,id,int32,string,0


## 5. Standardize text

Boundary whitespace and line endings are normalized; symbol/form codes are uppercased. Internal narrative text, HTML entities, and identifier leading zeros are preserved. The preview below abbreviates long fields; the CSV retains full text.

In [7]:
print('text_changes.csv')
display(evidence('text_changes.csv').head(12))

text_changes.csv


,table,source_row,column,before,after
0,filings,0,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
1,filings,1,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
2,filings,2,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
3,filings,3,mda_content,Item 7. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 7. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
4,filings,4,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
5,filings,5,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
6,filings,6,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
7,filings,7,mda_content,Item 7. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 7. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
8,filings,8,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...
9,filings,9,mda_content,ITEM 7. MANAGEMENT&#8217;S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERAT...,ITEM 7. MANAGEMENT&#8217;S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATI...


## 6. Investigate duplicates

Exact duplicate rows, repeated IDs, and logical keys are counted separately. Policies and actual counts are recorded by the script.

In [8]:
print('duplicates.csv')
display(evidence('duplicates.csv'))

duplicates.csv


,table,exact_extra_rows,raw_repeated_id_rows,scoped_repeated_id_rows,scoped_logical_duplicate_rows,scoped_logical_duplicate_groups
0,prices,0,0,0,0,0
1,news,0,0,0,0,0
2,filings,0,0,0,0,0


## 7. Handle missing data

No price or text imputation is performed. Original-field missing counts are separate from expected missing values in derived returns.

In [9]:
print('missing_before_after.csv')
display(evidence('missing_before_after.csv'))

missing_before_after.csv


,table,column,before,after,before_rows,after_rows
0,prices,id,0,0,2656,2324
1,prices,symbol,0,0,2656,2324
2,prices,date,0,0,2656,2324
3,prices,open,0,0,2656,2324
4,prices,high,0,0,2656,2324
5,prices,low,0,0,2656,2324
6,prices,close,0,0,2656,2324
7,prices,adj_close,0,0,2656,2324
8,prices,volume,0,0,2656,2324
9,news,id,0,0,3888,3402


## 8. Check suspicious values

Domain violations may be set missing and flagged; unusual observations are reviewed rather than automatically corrected. Zero flags do not prove the source is error-free.

In [10]:
print('suspicious.csv')
display(evidence('suspicious.csv'))

suspicious.csv


,table,check,count,action
0,prices,flag_invalid_numeric,0,keep flagged; invalid numeric cells set missing
1,prices,flag_ohlc,0,keep flagged; invalid numeric cells set missing
2,prices,flag_zero_volume,0,keep flagged; invalid numeric cells set missing
3,prices,flag_large_return,0,keep flagged; invalid numeric cells set missing
4,prices,flag_date_gap,0,keep flagged; invalid numeric cells set missing
5,prices,flag_future_date,0,keep flagged; invalid numeric cells set missing
6,news,flag_future_date,0,keep flagged; invalid numeric cells set missing
7,news,flag_missing_text,0,keep flagged; invalid numeric cells set missing
8,filings,flag_future_date,0,keep flagged; invalid numeric cells set missing
9,filings,flag_form,0,keep flagged; invalid numeric cells set missing


In [11]:
print('numeric_changes.csv')
display(evidence('numeric_changes.csv'))

numeric_changes.csv


,table,source_row,column,before,after


## 9. Verify merges

Events are aggregated by symbol/date before validated one-to-one left joins. Counts and unmatched keys cover both sides. These are same-date coverage checks, not temporal alignment for subsequent news effects. Unmatched event files preserve the evidence.

In [12]:
print('merge_audit.csv')
display(evidence('merge_audit.csv'))

merge_audit.csv


,table,keys,relationship,before,after,extra_rows,price_keys_without_event,event_keys_without_price,matched_keys
0,news,"symbol,date",one_to_one after aggregation,2324,2324,0,0,1078,2324
1,filings,"symbol,date",one_to_one after aggregation,2324,2324,0,2289,28,35


## 10. Reconcile changes

Exclusion reasons account for the change in row counts. Volume reconciliation checks that cleaning and joins do not silently change totals. Summed share volume is an accounting check, not dollar value.

In [13]:
print('row_reconciliation.csv')
display(evidence('row_reconciliation.csv'))

row_reconciliation.csv


,table,start,exact_duplicate,invalid_key,scope,conflicting_price_key,final
0,prices,2656,0,0,332,0,2324
1,news,3888,0,0,486,0,3402
2,filings,73,0,0,10,0,63


In [14]:
print('volume_reconciliation.csv')
display(evidence('volume_reconciliation.csv'))

volume_reconciliation.csv


,stage,rows,volume
0,scope,332,1.454422e+09
1,retained_before_domain_cleaning,2324,1.580024e+11
2,invalid_volume_removed_from_retained,0,0.000000e+00
3,cleaned,2324,1.580024e+11
4,merged,2324,1.580024e+11


## 11. Check calculations and the Codex suggestion

The script calculates within-company current/previous observed adjusted close minus one without forward filling. It independently checks every return with Decimal and tests an explicitly artificial missing-value sequence. Review the PASS message above. Means use nonmissing observations. These checks do not yet validate news-aligned forward returns or rolling volume/volatility metrics.

In [15]:
print('metrics.csv')
display(evidence('metrics.csv'))

metrics.csv


,symbol,rows,valid_returns,mean_return,valid_volume,mean_volume,volume_sum,missing_returns
0,AAPL,332,331,0.000372,332,5.256276e+07,1.745083e+10,1
1,AMZN,332,331,0.000187,332,4.513671e+07,1.498539e+10,1
2,GOOGL,332,331,0.001776,332,3.557789e+07,1.181186e+10,1
3,META,332,331,0.000193,332,1.538954e+07,5.109327e+09,1
4,MSFT,332,331,-0.000297,332,2.475262e+07,8.217871e+09,1
5,NVDA,332,331,0.001127,332,2.118627e+08,7.033842e+10,1
6,TSLA,332,331,0.000820,332,9.062856e+07,3.008868e+10,1


## 12. Verify five individual records

The script declares expectations before cleaning, then compares original/expected/actual values. CSV blank values for the first return mean missing, not zero. Full strings are checked even when the display is shortened.

In [16]:
print('record_check_plan.csv')
display(evidence('record_check_plan.csv'))

record_check_plan.csv


,table,source_row,field,original,expected,reason
0,prices,332,row_presence,ADBE,excluded:scope,ADBE outside MAG7
1,prices,0,daily_return,238.28829956054688,,First observed price for this symbol has no prior denominator; return must be missing
2,prices,1,date,2024-12-03,2024-12-03,Preserve calendar date
3,news,0,highlights,"Apple’s December 1, 2024 coverage centers on its fiscal first quarter 2024 results, with Shackne...","Apple’s December 1, 2024 coverage centers on its fiscal first quarter 2024 results, with Shackne...",Boundary/newline normalization; preserve internal text
4,filings,0,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...,Normalize actual filing text boundaries and line endings without changing internal content


In [17]:
print('record_checks.csv')
display(evidence('record_checks.csv'))

record_checks.csv


,table,source_row,field,original,expected,reason,actual,pass
0,prices,332,row_presence,ADBE,excluded:scope,ADBE outside MAG7,excluded:scope,True
1,prices,0,daily_return,238.28829956054688,,First observed price for this symbol has no prior denominator; return must be missing,,True
2,prices,1,date,2024-12-03,2024-12-03,Preserve calendar date,2024-12-03,True
3,news,0,highlights,"Apple’s December 1, 2024 coverage centers on its fiscal first quarter 2024 results, with Shackne...","Apple’s December 1, 2024 coverage centers on its fiscal first quarter 2024 results, with Shackne...",Boundary/newline normalization; preserve internal text,"Apple’s December 1, 2024 coverage centers on its fiscal first quarter 2024 results, with Shackne...",True
4,filings,0,mda_content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operat...,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...,Normalize actual filing text boundaries and line endings without changing internal content,Item 2. Management&#8217;s Discussion and Analysis of Financial Condition and Results of Operati...,True


## 13. Test reproducibility and inspect the sample

The first run establishes a hash baseline; a fresh-kernel rerun should print `PASS: outputs match previous complete run.` The checklist records the script's status. Review the exact environment recorded below and align submission dependencies with it.

The script creates `cleaning_outputs/cleaned_sample_max50.csv` containing at most 50 rows total. Selection prioritizes declared checks, difficult cases when present, company coverage, and stable-order context. This is a purposive review sample, not a representative statistical sample. Keep the original generated sample in place; copy it elsewhere if needed for Git submission.

In [18]:
display(pd.DataFrame([json.loads((OUTPUT_DIR / 'environment.json').read_text('utf-8'))]))
sample = evidence('cleaned_sample_max50.csv')
assert len(sample) <= 50
print('Sample:', OUTPUT_DIR / 'cleaned_sample_max50.csv')
print('Sample rows:', len(sample))
display(sample[['table', 'sample_reason', 'symbol', 'date', '_source_row']])
display(evidence('checklist_14.csv'))

,python,packages,source_version
0,3.11.9,"{'pandas': '2.3.2', 'numpy': '2.2.6', 'pyarrow': '22.0.0'}",Local snapshot; identify exact files by SHA256. Upstream revision not assumed.


Sample: C:\Users\yexun\OneDrive\Desktop\Python-Group-Project\cleaning_outputs\cleaned_sample_max50.csv
Sample rows: 50


,table,sample_reason,symbol,date,_source_row
0,prices,predeclared record check,AAPL,2024-12-02,0
1,prices,predeclared record check,AAPL,2024-12-03,1
2,prices,symbol coverage / first observed row,AMZN,2024-12-02,664
3,prices,symbol coverage / first observed row,GOOGL,2024-12-02,996
4,prices,symbol coverage / first observed row,META,2024-12-02,1328
5,prices,symbol coverage / first observed row,MSFT,2024-12-02,1660
6,prices,symbol coverage / first observed row,NVDA,2024-12-02,1992
7,prices,symbol coverage / first observed row,TSLA,2024-12-02,2324
8,prices,stable-order context,AAPL,2024-12-04,2
9,prices,stable-order context,AAPL,2024-12-05,3


,item,requirement,status,evidence,explanation
0,1,State the project vision and question,Done,Section 1,
1,2,Define one row,Done,Section 2; duplicates.csv,
2,3,Preserve the source,Done,local_source_manifest.json; raw file hashes,
3,4,Inspect columns and types,Done,type_checks.csv,
4,5,Standardize text,Done,text_changes.csv (zero changes is valid),
5,6,Investigate duplicates,Done,duplicates.csv; excluded_*.csv; logical_duplicates_*.csv,
6,7,Handle missing data,Done,missing_before_after.csv,
7,8,Check suspicious values,Done,suspicious.csv; flagged_*.csv,
8,9,Verify merges,Done,merge_audit.csv; unmatched_*_keys.csv,
9,10,Reconcile changes,Done,row_reconciliation.csv; volume_reconciliation.csv,


## 14. Document remaining limitations

- Seven large technology companies and a limited historical window restrict generalization; this snapshot cannot prioritize research for the current week.
- Daily news digests are summaries, not original articles. Every observed price day has news; content classification is needed for meaningful event comparisons.
- Publication timestamps and a verified exchange calendar are unavailable. Same-date joins do not establish temporal ordering. Subsequent-trading-day mapping, weekend-news treatment, and rolling windows remain to be defined and tested.
- Filing excerpts are partial and adjusted-price methodology has not been independently verified. No flags under the configured checks is not proof of completeness or accuracy.
- Missing or quarantined observations and upstream duplicates could bias coverage. Group means are descriptive and do not establish causality.

The 14-item checklist describes cleaning and verification evidence, not completion of the downstream research study. Submit the script, this executed notebook, a sample, dependency files, and README on your individual branch. Your separate report and README should carry matching branch and analysis commit information. Update them to reflect the latest actual run.